# Amaliyot 3. Multiclass va multilabel

3-interaktiv laboratoriyaga («Multiclass va multilabel») hamroh. Kurs mavzusi: koʻp sinfli va koʻp yorliqli tasniflash modellari

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.3, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

## 1. Multiclass: bitta sinf, ehtimollar yigʻindisi 1

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, classification_report

vec = TfidfVectorizer()
Xa = vec.fit_transform(df.text)
mc = LogisticRegression(max_iter=1000).fit(Xa, df.label)
q = vec.transform(["oʻyin jamoa gol byudjet qonun deputat"])
p = mc.predict_proba(q)[0]
print(dict(zip(mc.classes_, p.round(3))), "yigʻindi =", p.sum().round(3))

## 2. Multilabel: hujjatda bir nechta yorliq boʻlishi mumkin
Mavzular tutashgan joyda hujjatlar yaratamiz: turli mavzulardagi ikki matnni biriktiramiz, yorliqlar ikkalasi.

In [ ]:
rng = np.random.default_rng(0)
labs = sorted(df.label.unique())
pool = {l: df[df.label == l].text.tolist() for l in labs}
rows = []
for _ in range(300):
    a, b = rng.choice(labs, 2, replace=False)
    rows.append((rng.choice(pool[a]) + " " + rng.choice(pool[b]), [a, b]))
multi = pd.concat([pd.DataFrame({"text": df.text, "labels": [[l] for l in df.label]}),
                   pd.DataFrame(rows, columns=["text", "labels"])], ignore_index=True)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(multi.labels)
Xm = vec.fit_transform(multi.text)
Xtr, Xte, Ytr, Yte = train_test_split(Xm, Y, test_size=0.3, random_state=0)
ovr = OneVsRestClassifier(LogisticRegression(C=10, max_iter=1000)).fit(Xtr, Ytr)
P = ovr.predict_proba(Xte)
print("Sinflar:", mlb.classes_)
print("Ehtimollar misoli (yigʻindisi 1 emas):", P[0].round(2), "yigʻindi =", P[0].sum().round(2))

In [ ]:
pred = (P >= 0.5).astype(int)
print("Yorliqlar toʻplamining toʻliq mosligi:", round(accuracy_score(Yte, pred), 3))
print("Hamming loss:", round(hamming_loss(Yte, pred), 3))
print("F1 micro:", round(f1_score(Yte, pred, average="micro"), 3))
print(classification_report(Yte, pred, target_names=mlb.classes_, zero_division=0))

In [ ]:
ths = np.linspace(0.05, 0.95, 19)
plt.plot(ths, [f1_score(Yte, (P >= t).astype(int), average="micro") for t in ths], marker="o")
plt.xlabel("yorliq chegarasi"); plt.ylabel("F1 micro"); plt.title("Multilabel uchun chegarani tanlash"); plt.show()

## Topshiriqlar

1. Har bir yorliq uchun alohida optimal chegarani tanlang. Individual chegara nima beradi?
2. Solishtiring `OneVsRestClassifier` bilan `LogisticRegression` (multinomial) ni shu maʼlumotlarning multiclass-versiyasida (har bir hujjatga bitta yorliq qoldiring).
3. Kompyuter lingvistikasidan multilabel masalasiga oʻz misolingizni keltiring va u uchun metrika taklif qiling.